# GEOCODER_BATCH_Generatore_di_Coordinate

## INSTALLAZIONE LIBRERIE

In [8]:
!pip install geopy

## IMPORTAZIONE LIBRERIE

In [16]:
import pandas as pd
import json, time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

## CODE GREZZO


In [46]:
import pandas as pd
import json, time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

SLEEP_SEC = 1.1 # Nominatim: max 1 req/sec
CHECKPOINT = 50 # salva ogni N righe
USER_AGENT = 'gis_portfolio_geocoder/1.0'

def geocode_address(geocoder, address: str, retries=3):
    for attempt in range(retries):
        try:
            loc = geocoder.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except GeocoderTimedOut:
            time.sleep(2 ** attempt) # backoff esponenziale
        except GeocoderServiceError as e:
            print(f' Errore servizio: {e}')
            return (None, None)
    return (None, None)

def run_batch(input_path: str, address_col: str):
    # Sistemato l'a capo
    if input_path.endswith('.csv'):
        df = pd.read_csv(input_path)
    else:
        df = pd.read_excel(input_path)
        
    geocoder = Nominatim(user_agent=USER_AGENT)
    results = []
    failed = []
    out_stem = Path(input_path).stem # Corretto .steam in .stem

    for i, row in df.iterrows(): # Corretto .interrows() in .iterrows()
        addr = str(row[address_col])
        lat, lon = geocode_address(geocoder, addr)
        
        if lat is not None:
            results.append({**row.to_dict(), 'lat': lat, 'lon': lon})
        else:
            # Aggiunta parentesi di chiusura
            failed.append({**row.to_dict(), 'errore': 'non trovato'})
            print(f' [FAIL] {addr}')
            
        time.sleep(SLEEP_SEC)

        # Checkpoint ogni N righe
        if (i + 1) % CHECKPOINT == 0:
            pd.DataFrame(results).to_csv(f'{out_stem}_checkpoint.csv', index=False)
            print(f' Checkpoint salvato ({i+1} righe)')

    # Output finale
    df_ok = pd.DataFrame(results)
    df_ok.to_csv(f'{out_stem}_geocoded.csv', index=False) # Aggiunta parentesi
    
    if failed:
        # Aggiunta virgola
        pd.DataFrame(failed).to_csv(f'{out_stem}_failed.csv', index=False)

    # GeoJSON per ArcGIS Pro - Corretti 'coordinates', formato [lon, lat] e r.items()
    features = [
        {
            'type':'Feature',
            'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
            'properties': {k: v for k, v in r.items() if k not in ('lat', 'lon')}
        }
        for r in results
    ]
    
    with open(f'{out_stem}_geocoded.geojson', 'w') as f:
        json.dump({'type':'FeatureCollection', 'features': features}, f, ensure_ascii=False)

    # Corretta parentesi finale
    print(f'\nGeocoded: {len(results)} | Failed: {len(failed)}')
    return df_ok

if __name__ == '__main__':
    path = input('File CSV/XLSX: ').strip()
    col = input('Colonna indirizzi: ').strip()
    run_batch(path, col)

File CSV/XLSX:  C:\Users\bocci\OneDrive\Desktop\Indirizzi_Test_Italia.xlsx
Colonna indirizzi:  Indirizzo_Completo


 [FAIL] Via Non Esistente 1234, 00000 Posto Fantasma

Geocoded: 29 | Failed: 1


In [ ]:
C:\Users\bocci\OneDrive\Desktop\Indirizzi_Test_Italia.xlsx

In [ ]:
Indirizzo_Completo

## CODE INCREMENTATO

In [65]:
import pandas as pd
import json, time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

SLEEP_SEC = 1.1 # Nominatim: max 1 req/sec
CHECKPOINT = 50 # salva ogni N righe
USER_AGENT = 'gis_portfolio_geocoder/1.0'

def geocode_address(geocoder, address: str, retries=3):
    for attempt in range(retries):
        try:
            loc = geocoder.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except GeocoderTimedOut:
            time.sleep(2 ** attempt) # backoff esponenziale
        except GeocoderServiceError as e:
            print(f' Errore servizio: {e}')
            return (None, None)
    return (None, None)

def run_batch(input_path: str, col_addr: str, col_city: str, col_cap: str):
    if input_path.endswith('.csv'):
        df = pd.read_csv(input_path, dtype=str) # Legge tutto come stringa per non rovinare i CAP
    else:
        df = pd.read_excel(input_path, dtype=str)
        
    geocoder = Nominatim(user_agent=USER_AGENT)
    results = []
    failed = []
    out_stem = Path(input_path).stem

    for i, row in df.iterrows():
        # 1. Costruiamo l'indirizzo unendo i vari pezzi (se presenti)
        addr_parts = [str(row[col_addr])]
        
        # Aggiungiamo il CAP se l'utente ha inserito la colonna e la cella non è vuota
        if col_cap and col_cap in df.columns and pd.notna(row[col_cap]):
            addr_parts.append(str(row[col_cap]))
            
        # Aggiungiamo la Città se l'utente ha inserito la colonna e la cella non è vuota
        if col_city and col_city in df.columns and pd.notna(row[col_city]):
            addr_parts.append(str(row[col_city]))

        # Uniamo tutti i pezzi con una virgola e spazio
        full_address = ", ".join(addr_parts)
        
        # 2. Geocodifica
        lat, lon = geocode_address(geocoder, full_address)
        
        if lat is not None:
            # Salviamo anche l'indirizzo ricostruito per verifica
            results.append({**row.to_dict(), 'Indirizzo_Cercato': full_address, 'lat': lat, 'lon': lon})
        else:
            failed.append({**row.to_dict(), 'Indirizzo_Cercato': full_address, 'errore': 'non trovato'})
            print(f' [FAIL] {full_address}')
            
        time.sleep(SLEEP_SEC)

        # Checkpoint ogni N righe
        if (i + 1) % CHECKPOINT == 0:
            pd.DataFrame(results).to_csv(f'{out_stem}_checkpoint.csv', index=False)
            print(f' Checkpoint salvato ({i+1} righe)')

    # Output finale
    df_ok = pd.DataFrame(results)
    df_ok.to_csv(f'{out_stem}_geocoded.csv', index=False)
    
    if failed:
        pd.DataFrame(failed).to_csv(f'{out_stem}_failed.csv', index=False)

    # GeoJSON per ArcGIS Pro
    features = [
        {
            'type':'Feature',
            'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
            'properties': {k: v for k, v in r.items() if k not in ('lat', 'lon')}
        }
        for r in results
    ]
    
    with open(f'{out_stem}_geocoded.geojson', 'w') as f:
        json.dump({'type':'FeatureCollection', 'features': features}, f, ensure_ascii=False)

    print(f'\n✅ Finito! Geocodificati: {len(results)} | Falliti: {len(failed)}')
    return df_ok

if __name__ == '__main__':
    print("=== GEOCODER GIS BATCH ===")
    path = input('File CSV/XLSX: ').strip()
    
    print("\n--- IMPOSTAZIONI COLONNE ---")
    col_addr = input('1. Colonna Indirizzo (es. via Roma 1): ').strip()
    print("\n(Premi semplicemente INVIO se non hai le prossime colonne separate)")
    col_cap = input('2. Colonna CAP (opzionale): ').strip()
    col_city = input('3. Colonna Città (opzionale): ').strip()
    
    print("\nElaborazione in corso, attendere...\n")
    run_batch(path, col_addr, col_city, col_cap)

=== GEOCODER GIS BATCH ===


File CSV/XLSX:  E:\PYTHON - GIS\9_algoritmi_Portfolio\ALGORITMO 2 — Geocoder Batch da Indirizzo a Coordinate\StressTest_Solo_Vie.xlsx



--- IMPOSTAZIONI COLONNE ---


1. Colonna Indirizzo (es. via Roma 1):  Indirizzo



(Premi semplicemente INVIO se non hai le prossime colonne separate)


2. Colonna CAP (opzionale):  
3. Colonna Città (opzionale):  



Elaborazione in corso, attendere...


✅ Finito! Geocodificati: 8 | Falliti: 0


In [ ]:
Indirizzo_Completo

In [ ]:
Indirizzo_Completo

In [ ]:
Città

## CODE IMPLEMENTATO

In [11]:
import pandas as pd
import json, time
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

SLEEP_SEC = 1.1 # Nominatim: max 1 req/sec
CHECKPOINT = 50 # salva ogni N righe
USER_AGENT = 'gis_portfolio_geocoder_v7/1.0'

# Lista completa di tutti i capoluoghi di provincia italiani
CAPOLUOGHI = {
    'agrigento', 'alessandria', 'ancona', 'aosta', 'arezzo', 'ascoli piceno', 'asti', 
    'avellino', 'bari', 'barletta', 'andria', 'trani', 'belluno', 'benevento', 'bergamo', 
    'biella', 'bologna', 'bolzano', 'brescia', 'brindisi', 'cagliari', 'caltanissetta', 
    'campobasso', 'carbonia', 'caserta', 'catania', 'catanzaro', 'chieti', 'como', 
    'cosenza', 'cremona', 'crotone', 'cuneo', 'enna', 'fermo', 'ferrara', 'firenze', 
    'foggia', 'forlì', 'cesena', 'frosinone', 'genova', 'gorizia', 'grosseto', 'imperia', 
    'isernia', "l'aquila", 'la spezia', 'latina', 'lecce', 'lecco', 'livorno', 'lodi', 
    'lucca', 'macerata', 'mantova', 'massa', 'carrara', 'matera', 'messina', 'milano', 
    'modena', 'monza', 'napoli', 'novara', 'nuoro', 'oristano', 'padova', 'palermo', 
    'parma', 'pavia', 'perugia', 'pesaro', 'urbino', 'pescara', 'piacenza', 'pisa', 
    'pistoia', 'pordenone', 'potenza', 'prato', 'ragusa', 'ravenna', 'reggio calabria', 
    'reggio emilia', 'rieti', 'rimini', 'roma', 'rovigo', 'salerno', 'sassari', 'savona', 
    'siena', 'siracusa', 'sondrio', 'taranto', 'teramo', 'terni', 'torino', 'trapani', 
    'trento', 'treviso', 'trieste', 'udine', 'varese', 'venezia', 'verbania', 'vercelli', 
    'verona', 'vibo valentia', 'vicenza', 'viterbo'
}

def geocode_address(geocoder, address: str, retries=3):
    for attempt in range(retries):
        try:
            loc = geocoder.geocode(address, timeout=10, addressdetails=True)
            if loc:
                addr_info = loc.raw.get('address', {})
                country_code = addr_info.get('country_code', '').lower()
                
                citta_raw = addr_info.get('city', addr_info.get('town', addr_info.get('village', addr_info.get('county', ''))))
                nome_citta = citta_raw.title() if citta_raw else "Non determinata"
                postcode = addr_info.get('postcode', '')
                city_name_lower = citta_raw.lower() if citta_raw else ""
                is_city_tag = 'city' in addr_info
                
                # Classificazione
                if country_code != 'it':
                    categoria = 'Estero'
                elif is_city_tag or city_name_lower in CAPOLUOGHI:
                    categoria = 'Capoluogo/Grande Città'
                else:
                    categoria = 'Provincia/Piccolo Comune'
                    
                return (loc.latitude, loc.longitude, categoria, nome_citta, postcode)
            
            return (None, None, None, None, None)
        except GeocoderTimedOut:
            time.sleep(2 ** attempt)
        except GeocoderServiceError as e:
            print(f' Errore servizio: {e}')
            return (None, None, None, None, None)
    return (None, None, None, None, None)

def colora_matrix(df_export, df_completo):
    """Crea una matrice di colori della stessa forma del dataframe esportato"""
    styles = pd.DataFrame('', index=df_export.index, columns=df_export.columns)
    
    for idx in df_export.index:
        # Recuperiamo la riga originale con i dati nascosti
        riga = df_completo.loc[idx]
        cat = riga.get('Categoria_Geografica')
        bg = ''
        
        # Colore base riga
        if cat == 'Estero':
            bg = 'background-color: #FF9999' # Rosso chiaro
        elif cat == 'Capoluogo/Grande Città':
            bg = 'background-color: #90EE90' # Verde chiaro
        elif cat == 'Provincia/Piccolo Comune':
            bg = 'background-color: #FFFFE0' # Giallo chiaro
            
        styles.loc[idx, :] = bg
        
        # Colore specifico celle riparate
        imputed_str = str(riga.get('_imputed', ''))
        if imputed_str and imputed_str != 'nan':
            for col in imputed_str.split(','):
                if col and col in styles.columns:
                    styles.loc[idx, col] = 'background-color: #87CEFA' # Azzurro
                    
    return styles

def run_batch(input_path: str, col_id: str, col_addr: str, col_city: str, col_cap: str):
    input_path = input_path.strip('"').strip("'")
    
    df = pd.read_csv(input_path, dtype=str) if input_path.endswith('.csv') else pd.read_excel(input_path, dtype=str)
        
    geocoder = Nominatim(user_agent=USER_AGENT)
    results, failed = [], []
    out_stem = Path(input_path).stem
    out_dir = Path(input_path).parent

    for i, row in df.iterrows():
        addr_parts = [str(row[col_addr])]
        if col_cap and col_cap in df.columns and pd.notna(row[col_cap]):
            addr_parts.append(str(row[col_cap]))
        if col_city and col_city in df.columns and pd.notna(row[col_city]):
            addr_parts.append(str(row[col_city]))

        full_address = ", ".join(addr_parts)
        lat, lon, categoria, nome_citta, postcode = geocode_address(geocoder, full_address)
        
        row_dict = row.to_dict()
        output_row = {}
        imputed_cols = [] # Tracker per le celle azzurre
        
        if col_id and col_id in df.columns:
            output_row[col_id] = row_dict.pop(col_id)
            
        output_row.update(row_dict)
        
        if lat is not None:
            # --- LOGICA DI RIEMPIMENTO DATI MANCANTI ---
            if col_cap and col_cap in df.columns:
                val_cap = str(output_row.get(col_cap, '')).strip().lower()
                if val_cap in ('', 'nan', 'none') and postcode:
                    output_row[col_cap] = postcode
                    imputed_cols.append(col_cap)
                    print(f'   [FIX] Aggiunto CAP ({postcode}) in {full_address}')
            
            if col_city and col_city in df.columns:
                val_city = str(output_row.get(col_city, '')).strip().lower()
                if val_city in ('', 'nan', 'none') and nome_citta != "Non determinata":
                    output_row[col_city] = nome_citta
                    imputed_cols.append(col_city)
                    print(f'   [FIX] Aggiunta Città ({nome_citta}) in {full_address}')
            
            # --------------------------------------------
            
            output_row.update({
                'Indirizzo_Cercato': full_address, 
                'Città_Trovata': nome_citta,
                'lat': lat, 'lon': lon,
                'Categoria_Geografica': categoria,
                '_imputed': ",".join(imputed_cols) # Colonna nascosta per i colori
            })
            results.append(output_row)
        else:
            output_row.update({'Indirizzo_Cercato': full_address, 'errore': 'non trovato'})
            failed.append(output_row)
            print(f' [FAIL] {full_address}')
            
        time.sleep(SLEEP_SEC)

    # --- SALVATAGGIO IN UN UNICO FILE EXCEL CON DUE FOGLI ---
    out_excel = out_dir / f'{out_stem}_risultati_finali.xlsx'
    
    with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:
        df_completo = pd.DataFrame(results)
        
        if not df_completo.empty:
            # Rimuoviamo la colonna nascosta per non sporcare il file Excel finale
            df_export = df_completo.drop(columns=['_imputed'])
            
            # Applichiamo i colori usando la matrice
            df_styled = df_export.style.apply(lambda x: colora_matrix(df_export, df_completo), axis=None)
            df_styled.to_excel(writer, sheet_name='Geocodificati', index=False)
            print(f'\n✅ Foglio "Geocodificati" creato.')
        
        df_fail = pd.DataFrame(failed)
        if not df_fail.empty:
            df_fail.to_excel(writer, sheet_name='Falliti', index=False)
            print(f'⚠️ Foglio "Falliti" aggiunto al file.')

    if results:
        features = [
            {
                'type':'Feature',
                'geometry': {'type': 'Point', 'coordinates': [r['lon'], r['lat']]},
                'properties': {k: v for k, v in r.items() if k not in ('lat', 'lon', 'Categoria_Geografica', '_imputed')}
            } for r in results
        ]
        with open(out_dir / f'{out_stem}_geocoded.geojson', 'w', encoding='utf-8') as f:
            # Aggiunto indent=4 per formattare il file in verticale e renderlo leggibile
            json.dump({'type':'FeatureCollection', 'features': features}, f, ensure_ascii=False, indent=4)

    print(f'\n📊 Elaborazione completata: {out_excel.name}')

if __name__ == '__main__':
    print("=== GEOCODER GIS BATCH (V7: Imputation & Color Matrix) ===")
    print("\n---------------- LEGENDA COLORI ----------------")
    print("🟩 VERDE    -> Capoluogo o Grande Città Italiana")
    print("🟨 GIALLO   -> Provincia o Piccolo Comune Italiano")
    print("🟥 ROSSO    -> Estero (Internazionale)")
    print("🟦 AZZURRO  -> Dato mancante riparato dallo script")
    print("------------------------------------------------\n")
    
    path = input('File CSV/XLSX: ').strip().strip('"').strip("'")
    print("\n--- IMPOSTAZIONI COLONNE ---")
    c_id = input('0. Colonna ID Univoco (opzionale): ').strip()
    c_addr = input('1. Colonna Indirizzo: ').strip()
    c_cap = input('2. Colonna CAP (opzionale): ').strip()
    c_city = input('3. Colonna Città (opzionale): ').strip()
    
    print("\nAvvio processo...\n")
    run_batch(path, c_id, c_addr, c_city, c_cap)

=== GEOCODER GIS BATCH (V7: Imputation & Color Matrix) ===

---------------- LEGENDA COLORI ----------------
🟩 VERDE    -> Capoluogo o Grande Città Italiana
🟨 GIALLO   -> Provincia o Piccolo Comune Italiano
🟥 ROSSO    -> Estero (Internazionale)
🟦 AZZURRO  -> Dato mancante riparato dallo script
------------------------------------------------



File CSV/XLSX:  "E:\PYTHON - GIS\9_algoritmi_Portfolio\ALGORITMO 2 — Geocoder Batch da Indirizzo a Coordinate\Dataset_Globale_Geocoder.csv"



--- IMPOSTAZIONI COLONNE ---


0. Colonna ID Univoco (opzionale):  ID
1. Colonna Indirizzo:  Indirizzo
2. Colonna CAP (opzionale):  CAP
3. Colonna Città (opzionale):  Città



Avvio processo...

   [FIX] Aggiunto CAP (80013) in Via Garibaldi 5, Napoli
   [FIX] Aggiunto CAP (50122) in Via Dante Alighieri 1
   [FIX] Aggiunta Città (Firenze) in Via Dante Alighieri 1
   [FIX] Aggiunta Città (Bologna) in Via Mazzini 20, 40137
   [FIX] Aggiunto CAP (95131) in Via Etnea 200, Catania
   [FIX] Aggiunto CAP (09041) in Via Indipendenza 10, Cagliari
   [FIX] Aggiunto CAP (90640) in Via Roma 1500
   [FIX] Aggiunta Città (Montebello) in Via Roma 1500
   [FIX] Aggiunta Città (Napoli) in Corso Umberto I 15, 80138
   [FIX] Aggiunto CAP (90018) in Via Libertà 50, Palermo
   [FIX] Aggiunto CAP (61030) in Viale Trieste 20, Pesaro
   [FIX] Aggiunto CAP (98056) in Corso Cavour 5, Messina
 [FAIL] Champ de Mars, 5 Av. Anatole France, 75007, Paris
   [FIX] Aggiunto CAP (150-0043) in Shibuya Crossing, Tokyo
   [FIX] Aggiunto CAP (1012 LG) in Damrak 1, Amsterdam
 [FAIL] 1 CN Tower, M5V 2T6, Toronto
 [FAIL] Hollywood Walk of Fame, 90028, Los Angeles
   [FIX] Aggiunto CAP (D02 C803) in